# Ejercicio: EDA Básico — Supermarket Sales (SOLUCIÓN)

**Diplomado en Data Science Aplicada con Python**  
Clase 8 — Práctica autónoma

---

## 1. Cargar el dataset

In [ ]:
import pandas as pd
import numpy as np

url = ("https://raw.githubusercontent.com/cmosquerat/"
       "arca-diplomado/main/clase-08/supermarket_sales.csv")
df = pd.read_csv(url)
df.head()

---
## 2. Estructura

In [ ]:
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nCada fila representa una transacción de venta en el supermercado.")

In [ ]:
df.tail()

---
## 3. Tipos de dato

In [ ]:
print(df.dtypes)

In [ ]:
num = df.select_dtypes(include="number")
print("Numéricas:", list(num.columns))

cat = df.select_dtypes(include="object")
print("Categóricas:", list(cat.columns))

`Date` aparece como `object` pero debería ser `datetime64`. Convertimos:

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Mes"] = df["Date"].dt.month

print("Tipo de Date:", df["Date"].dtype)
df[["Date", "Mes"]].head()

---
## 4. Datos faltantes (NaN)

In [ ]:
print("=== NaN por columna ===")
print(df.isna().sum())

print(f"\nTotal de NaN: {df.isna().sum().sum()}")

In [ ]:
# Este dataset no tiene NaN, pero si los tuviera:
# - Total (con outliers)  -> mediana
# - Rating (simétrica)    -> media
# - Payment (categórica)  -> moda

# df["Total"]   = df["Total"].fillna(df["Total"].median())
# df["Rating"]  = df["Rating"].fillna(df["Rating"].mean())
# df["Payment"] = df["Payment"].fillna(df["Payment"].mode()[0])

---
## 5. Exploración

In [ ]:
df.describe()

In [ ]:
print(f"Total promedio: ${df['Total'].mean():.2f}")
print(f"Total mediano:  ${df['Total'].median():.2f}")

In [ ]:
print("=== Transacciones por Product line ===")
print(df["Product line"].value_counts())

In [ ]:
print("=== Ingresos por Branch ===")
print(df.groupby("Branch")["Total"].sum().sort_values(ascending=False))

In [ ]:
print("=== Método de pago más usado ===")
print(df["Payment"].value_counts())

---
## 6. Visualización

Vamos a explorar distintos tipos de gráfica con seaborn.  
La idea es que investiguen la documentación: https://seaborn.pydata.org/api.html

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

### 6.1 Histograma de Total

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
sns.histplot(df["Total"], bins=25, color="#C82B40", ax=ax)
ax.set_title("Distribución del Total de Venta", fontsize=14, fontweight="bold")
ax.set_xlabel("Total ($)")
ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

### 6.2 Boxplot de Total por Branch

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x="Branch", y="Total",
            palette=["#C82B40", "#2563EB", "#16A34A"], ax=ax)
ax.set_title("Total por Sucursal", fontsize=14, fontweight="bold")
ax.set_xlabel("Sucursal")
ax.set_ylabel("Total ($)")
plt.tight_layout()
plt.show()

### 6.3 Conteo por Product line

`sns.countplot()` cuenta automáticamente cuántos registros hay por categoría.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
order = df["Product line"].value_counts().index
sns.countplot(data=df, y="Product line", order=order,
              palette="Set2", ax=ax)
ax.set_title("Transacciones por Línea de Producto", fontsize=14, fontweight="bold")
ax.set_xlabel("Cantidad")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 6.4 Barplot: Rating promedio por Product line

`sns.barplot()` calcula el promedio automáticamente. A diferencia de `countplot`, necesita un eje `y` numérico.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=df, y="Product line", x="Rating",
            palette="coolwarm", ax=ax)
ax.set_title("Rating Promedio por Línea de Producto", fontsize=14, fontweight="bold")
ax.set_xlabel("Rating promedio")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 6.5 Countplot con `hue`: Método de pago por género

El parámetro `hue` permite subdividir las barras por otra variable categórica.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df, x="Payment", hue="Gender",
              palette=["#C82B40", "#2563EB"], ax=ax)
ax.set_title("Método de Pago por Género", fontsize=14, fontweight="bold")
ax.set_xlabel("Método de Pago")
ax.set_ylabel("Cantidad")
ax.legend(title="Género")
plt.tight_layout()
plt.show()

### 6.6 Boxplot de Rating por Gender

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="Gender", y="Rating",
            palette="Set1", ax=ax)
ax.set_title("Rating por Género", fontsize=14, fontweight="bold")
ax.set_xlabel("Género")
ax.set_ylabel("Rating")
plt.tight_layout()
plt.show()

### 6.7 Múltiples gráficas en una figura

Con `plt.subplots(filas, columnas)` podemos organizar varias gráficas juntas.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.histplot(df["Rating"], bins=20, color="#2563EB", ax=axes[0])
axes[0].set_title("Distribución del Rating")

sns.boxplot(data=df, x="Branch", y="Rating", palette="Set2", ax=axes[1])
axes[1].set_title("Rating por Sucursal")

sns.countplot(data=df, x="Branch", palette="Set2", ax=axes[2])
axes[2].set_title("Transacciones por Sucursal")

plt.tight_layout()
plt.show()

---
## 7. Investigación: ¿Qué más puede hacer seaborn?

Busquen en la documentación de seaborn y prueben al menos **una gráfica nueva** que no hayamos visto.  

Ideas para investigar:
- `sns.violinplot()` — como un boxplot pero muestra la forma de la distribución
- `sns.stripplot()` — muestra cada punto individual
- `sns.pairplot()` — matriz de gráficas entre varias columnas
- `sns.kdeplot()` — curva suave de distribución (en vez de barras)

Documentación: https://seaborn.pydata.org/api.html

In [ ]:
# Ejemplo: violinplot
fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(data=df, x="Product line", y="Total",
               palette="Set2", ax=ax)
ax.set_title("Distribución de Total por Línea de Producto (Violin)",
             fontsize=14, fontweight="bold")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 8. Conclusiones

1. El dataset tiene 1,000 transacciones repartidas de forma bastante uniforme entre 3 sucursales y 6 líneas de producto.

2. No hay NaN en este dataset, pero practicamos la estrategia: mediana para numéricas con outliers, media para simétricas, moda para categóricas.

3. Las tres sucursales se comportan de manera similar en ventas y rating. Si fuera gerente, investigaría **qué factores diferencian** las sucursales más allá de los montos.